# N122 · 后缀数组与LCP

**目标：** 用后缀排序把重复子串问题变成相邻比较。

**先修：** N118, N037, N040。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 倍增秩；后缀数组；Kasai；LCP；不同子串计数；排序成本。

## 从问题到算法

把所有后缀按字典序排序后，每个不同子串都可以看成某个后缀的前缀。相邻后缀的最长公共前缀LCP告诉我们新后缀有多少前缀已经出现。倍增法每轮比较两个长度2^k块的排名，而不是反复比较整段字符串。

## 最小实现

**本章接口：** `suffix_array(s)`、`kasai_lcp(s, sa)`、`count_distinct_substrings_sa(s)`

In [1]:
def suffix_array(s):
    n=len(s); sa=list(range(n)); ranks=[ord(c) for c in s]; width=1
    while width<n:
        sa.sort(key=lambda i:(ranks[i],ranks[i+width] if i+width<n else -1))
        new=[0]*n
        for j in range(1,n):
            a,b=sa[j-1],sa[j]
            left=(ranks[a],ranks[a+width] if a+width<n else -1)
            right=(ranks[b],ranks[b+width] if b+width<n else -1)
            new[b]=new[a]+(left!=right)
        ranks=new
        if not n or ranks[sa[-1]]==n-1: break
        width*=2
    return sa

def kasai_lcp(s,sa):
    n=len(s)
    if len(sa)!=n or set(sa)!=set(range(n)): raise ValueError('sa must be a permutation')
    rank=[0]*n
    for r,i in enumerate(sa): rank[i]=r
    lcp=[0]*max(0,n-1); height=0
    for i in range(n):
        r=rank[i]
        if r==n-1: height=0; continue
        j=sa[r+1]
        while i+height<n and j+height<n and s[i+height]==s[j+height]: height+=1
        lcp[r]=height; height=max(0,height-1)
    return lcp

def count_distinct_substrings_sa(s):
    sa=suffix_array(s); return len(s)*(len(s)+1)//2-sum(kasai_lcp(s,sa))

## 正确性、前提与复杂度

两个长度2w片段的顺序由其前后两块长度w的排名决定，归纳得到完整后缀顺序。加入字典序下一后缀时，和所有旧后缀共享的最长前缀由相邻前驱给出；扣除相邻LCP总和即可去重。Kasai把前一匹配长度减1作为下一次下界，累计字符扩展线性。

**代价：** Python比较排序倍增O(n log² n)时间O(n)辅助，不误称O(n log n)。给定合法SA的Kasai O(n)时间空间；本实现额外线性验证排列，但不重新验证SA排序。非空不同子串数量为n(n+1)/2−∑LCP。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

s='banana'; sa=suffix_array(s); lcp=kasai_lcp(s,sa)
show_table(['排名','后缀起点','后缀','与下一后缀LCP'],[(r,i,s[i:],lcp[r] if r<len(lcp) else '-') for r,i in enumerate(sa)])
print('不同非空子串',count_distinct_substrings_sa(s))

排名,后缀起点,后缀,与下一后缀LCP
0,5,a,1
1,3,ana,3
2,1,anana,0
3,0,banana,0
4,4,na,2
5,2,nana,-


不同非空子串 15


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert suffix_array('banana')==[5,3,1,0,4,2]
assert suffix_array('')==[] and kasai_lcp('',[])==[]
from itertools import product
for n in range(8):
    for chars in product('ab',repeat=n):
        s=''.join(chars); sa=suffix_array(s); lc=kasai_lcp(s,sa)
        assert sa==sorted(range(n),key=lambda i:s[i:])
        assert count_distinct_substrings_sa(s)==len({s[i:j] for i in range(n) for j in range(i+1,n+1)})
        for r,h in enumerate(lc):
            a,b=s[sa[r]:],s[sa[r+1]:]
            assert h==max(k for k in range(min(len(a),len(b))+1) if a[:k]==b[:k])
print('N122: 所有本章断言通过')

N122: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 从LCP求最长重复子串。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 区分用比较排序的O(nlog²n)实现与计数排序版复杂度。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 1044. Longest Duplicate Substring（extension）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。